In [4]:
#!/usr/bin/env python3
"""
01_train_dnn_basic.py
=====================
Trains a lightweight Deep Neural Network (DNN / MLP) classifier for multi-class
sensory classification, applies full INT8 post-training quantization with a
representative dataset, and exports the model as a C++ header for tinyISB.
"""

'\n01_train_dnn_basic.py\n=====================\nTrains a lightweight Deep Neural Network (DNN / MLP) classifier for multi-class\nsensory classification, applies full INT8 post-training quantization with a\nrepresentative dataset, and exports the model as a C++ header for tinyISB.\n'

In [5]:
import os
import sys
import numpy as np
import tensorflow as tf
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [7]:
# Add parent directory to path to import exporter
sys.path.append(os.path.abspath(os.path.join(os.path.dirname("./"), "..")))
from tools.export_to_tinyisb import quantize_and_export_keras, inspect_tflite_bytes, print_model_info

In [8]:
print("[1/5] Generating synthetic sensory feature dataset (4 features, 3 classes)...")
np.random.seed(42)
tf.random.set_seed(42)

n_samples = 1500
n_features = 4
n_classes = 3

X, y = make_classification(
    n_samples=n_samples,
    n_features=n_features,
    n_informative=4,
    n_redundant=0,
    n_classes=n_classes,
    n_clusters_per_class=1,
    random_state=42
)

[1/5] Generating synthetic sensory feature dataset (4 features, 3 classes)...


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

print(f"      Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")
print(f"      Feature means: {scaler.mean_}")
print(f"      Feature scales: {scaler.scale_}")

      Train samples: 1200, Test samples: 300
      Feature means: [-0.27513404 -1.02272791  0.3682397  -0.98094074]
      Feature scales: [1.60097442 1.31210507 1.48402172 1.36888436]


In [10]:
print("\n[2/5] Constructing TinyML-optimized DNN architecture...")
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(n_features,), name="sensor_features"),
    tf.keras.layers.Dense(16, activation="relu", name="dense_1"),
    tf.keras.layers.Dense(8, activation="relu", name="dense_2"),
    tf.keras.layers.Dense(n_classes, activation="softmax", name="output_classes")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

print("\n[3/5] Training model...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=35,
    batch_size=32,
    verbose=0
)
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"      Training finished -> Test Accuracy: {test_acc * 100:.2f}%")


[2/5] Constructing TinyML-optimized DNN architecture...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_classes (Dense)          │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 243 (972.00 B)

 Trainable params: 243 (972.00 B)

 Non-trainable params: 0 (0.00 B)


[3/5] Training model...
      Training finished -> Test Accuracy: 86.33%


In [12]:
print("\n[4/5] Applying full INT8 post-training quantization with representative dataset...")
def representative_dataset_gen():
    for i in range(100):
        yield [X_train[i:i+1]]

output_header = os.path.abspath(os.path.join(
    os.path.dirname("./"), "..", "examples", "BasicClassification", "model_data.h"
))

tflite_bytes = quantize_and_export_keras(
    model,
    representative_data_gen=representative_dataset_gen,
    quantize_int8=True,
    output_header_path=output_header,
    array_name="model_data"
)


[4/5] Applying full INT8 post-training quantization with representative dataset...
INFO:tensorflow:Assets written to: C:\Users\User\AppData\Local\Temp\tmp0qmn305_\assets


INFO:tensorflow:Assets written to: C:\Users\User\AppData\Local\Temp\tmp0qmn305_\assets


Saved artifact at 'C:\Users\User\AppData\Local\Temp\tmp0qmn305_'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 4), dtype=tf.float32, name='sensor_features')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  2441017261520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2441017261712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2441017261136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2441017260944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2441017261904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2441017260368: TensorSpec(shape=(), dtype=tf.resource, name=None)


e:\PROJECTS\tinyISB\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
e:\PROJECTS\tinyISB\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
print(f"\n[5/5] Validating exported TFLite INT8 model...")
info = inspect_tflite_bytes(tflite_bytes)
print_model_info(info)

# Verify inference using TFLite interpreter
interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
interpreter.allocate_tensors()
inp_det = interpreter.get_input_details()[0]
out_det = interpreter.get_output_details()[0]

correct = 0
in_scale = inp_det["quantization"][0]
in_zero = inp_det["quantization"][1]
out_scale = out_det["quantization"][0]
out_zero = out_det["quantization"][1]

for i in range(len(X_test)):
    sample = X_test[i:i+1]
    # INT8 input scaling
    if inp_det["dtype"] == np.int8:
        q_input = np.round(sample / in_scale + in_zero).clip(-128, 127).astype(np.int8)
        interpreter.set_tensor(inp_det["index"], q_input)
    else:
        interpreter.set_tensor(inp_det["index"], sample)

    interpreter.invoke()
    q_out = interpreter.get_tensor(out_det["index"])[0]

    if out_det["dtype"] == np.int8:
        dequant_out = (q_out.astype(np.float32) - out_zero) * out_scale
    else:
        dequant_out = q_out

    pred_class = np.argmax(dequant_out)
    if pred_class == y_test[i]:
        correct += 1

quant_acc = correct / len(X_test)
print(f"      Quantized INT8 Model Accuracy: {quant_acc * 100:.2f}%")
print(f"      Header written to: {output_header}")

# Also print sample test input for Arduino verification
sample_feat = X_test[0]
sample_label = y_test[0]
print("\n--- Test Sample for Arduino Verification ---")
print(f"Input Features: {{{', '.join(f'{f:.4f}f' for f in sample_feat)}}}")
print(f"True Class: {sample_label}")
print(f"Predicted Class: {np.argmax(model.predict(X_test[0:1], verbose=0))}") 


[5/5] Validating exported TFLite INT8 model...
               tinyISB Model Inspection Report
Model Size: 3528 bytes (3.45 KB)

--- Inputs ---
 [0] serving_default_sensor_features:0
     Shape       : [1, 4]
     Data Type   : <class 'numpy.int8'>
     Quantized   : True (Scale=0.021663, ZeroPoint=-4)

--- Outputs ---
 [0] StatefulPartitionedCall_1:0
     Shape       : [1, 3]
     Data Type   : <class 'numpy.int8'>
     Quantized   : True (Scale=0.003906, ZeroPoint=-128)
      Quantized INT8 Model Accuracy: 86.00%
      Header written to: e:\PROJECTS\tinyISB\examples\BasicClassification\model_data.h

--- Test Sample for Arduino Verification ---
Input Features: {0.9375f, -0.3574f, 0.3241f, 0.4415f}
True Class: 2
Predicted Class: 2
